In [ ]:
import os
import sys
from pathlib import Path
import subprocess
import importlib
import json

# --- Environment Detection & Project Root Setup ---
IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = Path.cwd() # Default for local execution

if IN_COLAB:
    print("Running in Google Colab.")
    from google.colab import drive, output
    drive.mount('/content/drive', force_remount=True)
    # Default Colab project root (user should change this if their project is elsewhere)
    # This assumes the notebook itself is run from 'notebooks' directory within the project
    # Adjust if the notebook is copied to the root of the project in Drive.
    # For example, if project is at /content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai
    # and this notebook is notebooks/runyoro_bible_asr_finetune.ipynb
    # then PROJECT_ROOT might be /content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai
    # User needs to set this in the config cell if this default is not correct.
    COLAB_DEFAULT_PROJECT_ROOT = Path("/content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai")
    if COLAB_DEFAULT_PROJECT_ROOT.exists():
         PROJECT_ROOT = COLAB_DEFAULT_PROJECT_ROOT
    else:
        # Fallback if default path doesn't exist, prompt user to set it.
        print(f"WARNING: Default Colab project root {COLAB_DEFAULT_PROJECT_ROOT} not found.")
        print("Please set your PROJECT_ROOT manually in the 'Configuration' cell if current path is incorrect.")
        # Assuming notebook is in 'notebooks' under project root for this fallback
        PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/open-runyoro-ai/notebooks").parent # Adjust if needed
    
    # Ensure PROJECT_ROOT is added to sys.path for module imports
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    os.chdir(PROJECT_ROOT) # Change current directory to project root in Colab
    print(f"Project Root set to: {PROJECT_ROOT}")
    print(f"Current Working Directory set to: {os.getcwd()}")
else:
    print("Running in a local Jupyter/Lab environment.")
    # If local, assume notebook is in 'notebooks' dir, so parent is project root
    PROJECT_ROOT = Path.cwd().parent 
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Project Root set to: {PROJECT_ROOT}")
    print(f"Current Working Directory: {os.getcwd()}")

# --- Install Dependencies ---
# Common dependencies
dependencies = [
    "requests", "tqdm", "soundfile", "ipywidgets", 
    "torch", "torchaudio", "speechbrain", "sentencepiece", "PyYAML"
]

print("\nInstalling dependencies...")
for dep in dependencies:
    try:
        importlib.import_module(dep.split("==")[0].split(">=")[0].split("<=")[0]) # Try importing base name
        print(f"{dep} is already installed.")
    except ImportError:
        print(f"Installing {dep}...")
        if IN_COLAB:
            # Suppress output for cleaner notebook, check for errors below
            pip_process = subprocess.run([sys.executable, "-m", "pip", "install", dep], capture_output=True, text=True)
            if pip_process.returncode != 0:
                print(f"Error installing {dep} in Colab: {pip_process.stderr}")
            else:
                print(f"Successfully installed {dep} in Colab.")
        else:
            # For local, show output directly
            subprocess.check_call([sys.executable, "-m", "pip", "install", dep])

if IN_COLAB:
    print("\nInstalling libsndfile1 for soundfile to work correctly in Colab...")
    # Check if libsndfile is already installed to avoid unnecessary apt-get calls
    libsndfile_check = subprocess.run(["dpkg", "-s", "libsndfile1"], capture_output=True, text=True)
    if "Status: install ok installed" not in libsndfile_check.stdout:
        apt_process = subprocess.run(["apt-get", "update"], capture_output=True, text=True)
        if apt_process.returncode != 0: print(f"apt-get update failed: {apt_process.stderr}")
        apt_install_process = subprocess.run(["apt-get", "install", "-y", "libsndfile1"], capture_output=True, text=True)
        if apt_install_process.returncode != 0:
            print(f"Error installing libsndfile1 in Colab: {apt_install_process.stderr}")
        else:
            print("Successfully installed libsndfile1.")
    else:
        print("libsndfile1 is already installed.")

print("\nDependency installation check complete.")

# Verify SpeechBrain import
try:
    import speechbrain as sb
    print(f"SpeechBrain version: {sb.__version__}")
except ImportError as e:
    print(f"Failed to import SpeechBrain: {e}. Please check installation.")

# Re-check PROJECT_ROOT and ensure it's correct for module loading
# This is crucial if the user changes PROJECT_ROOT in the next cell
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT) # Ensure CWD is project root
print(f"Final Project Root: {PROJECT_ROOT}")
print(f"Final CWD: {os.getcwd()}")

In [ ]:
# --- Essential Configuration ---
# Option 1: If you cloned the repo and are running this notebook from within it:
# If running locally, PROJECT_ROOT is usually Path.cwd().parent if this notebook is in 'notebooks/'
# If in Colab, PROJECT_ROOT was set in the previous cell. You can override it here if necessary.
# Example: PROJECT_ROOT = Path("/content/drive/MyDrive/MyClonedRepo/open-runyoro-ai")

# Option 2: If you are running this notebook standalone and need to point to an existing clone:
# PROJECT_ROOT_OVERRIDE = "" # E.g., "/path/to/your/cloned/open-runyoro-ai"
# if PROJECT_ROOT_OVERRIDE:
#     PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE)
#     if str(PROJECT_ROOT) not in sys.path:
#          sys.path.insert(0, str(PROJECT_ROOT))
#     os.chdir(PROJECT_ROOT)
#     print(f"PROJECT_ROOT overridden to: {PROJECT_ROOT}")

# --- Script Paths (relative to PROJECT_ROOT) ---
DOWNLOAD_SCRIPT_PATH = PROJECT_ROOT / "runyoro_speech_ai" / "data_ingestion" / "download_bible_brain.py"
MANIFEST_SCRIPT_PATH = PROJECT_ROOT / "runyoro_speech_ai" / "asr_finetune" / "build_manifest.py"
TRAIN_SCRIPT_PATH = PROJECT_ROOT / "runyoro_speech_ai" / "asr_finetune" / "train_ctc.py"

# --- Default Data and Model Paths (can be overridden by widgets later) ---
# Note: data_dir will store downloaded audio/text and manifests
DEFAULT_DATA_DIR = PROJECT_ROOT / "data" / "bible_finetune_data" 
DEFAULT_SSL_CHECKPOINT = "/content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/colab_experiments/ssl_experiment_colab_01/checkpoints/checkpoint_15.ckpt"
# The ASR training script will save its checkpoints under PROJECT_ROOT / "asr_finetune" / "checkpoints" / HPARAMS_NAME
DEFAULT_ASR_OUTPUT_DIR = PROJECT_ROOT / "asr_finetune_output" # For hyperparams YAML and final model related to a run

# --- Verify script paths ---
assert DOWNLOAD_SCRIPT_PATH.exists(), f"Download script not found: {DOWNLOAD_SCRIPT_PATH}"
assert MANIFEST_SCRIPT_PATH.exists(), f"Manifest script not found: {MANIFEST_SCRIPT_PATH}"
assert TRAIN_SCRIPT_PATH.exists(), f"Train script not found: {TRAIN_SCRIPT_PATH}"

print(f"Using PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Download script: {DOWNLOAD_SCRIPT_PATH}")
print(f"Manifest script: {MANIFEST_SCRIPT_PATH}")
print(f"Train script: {TRAIN_SCRIPT_PATH}")
print(f"Default data directory: {DEFAULT_DATA_DIR}")
print(f"Default SSL checkpoint: {DEFAULT_SSL_CHECKPOINT}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

# --- UI Widgets ---
style = {'description_width': 'initial'}
layout_half = widgets.Layout(width='50%')
layout_full = widgets.Layout(width='90%')

api_key_input = widgets.Password(value=os.environ.get('DBP_API_KEY', ''), placeholder='Enter DBP API Key', description='DBP API Key:', style=style, layout=layout_half)
bible_id_input = widgets.Text(value='NYOBSN', placeholder='e.g., NYOBSN (Runyoro Audio Bible)', description='Bible ID (Audio Fileset):', style=style, layout=layout_half)
text_fileset_id_input = widgets.Text(value='NYOTBTN2ET', placeholder='e.g., NYOTBTN2ET (Runyoro Text)', description='Text Fileset ID:', style=style, layout=layout_half)
books_to_download_input = widgets.Text(value='MAT,MRK,LUK,JHN', placeholder='e.g., MAT,MRK,LUK,JHN (blank for all)', description='Books (comma-sep):', style=style, layout=layout_half)

data_dir_widget = widgets.Text(value=str(DEFAULT_DATA_DIR), description='Data Directory:', style=style, layout=layout_full)
ssl_ckpt_widget = widgets.Text(value=str(DEFAULT_SSL_CHECKPOINT), description='SSL Checkpoint Path:', style=style, layout=layout_full)
epochs_widget = widgets.IntText(value=5, description='Fine-tune Epochs:', style=style, layout=layout_half)

# Output directory for this specific ASR experiment run (contains hparams, tokenizer, saved model)
asr_experiment_name_widget = widgets.Text(value="runyoro_asr_ctc_finetune_v1", description="ASR Experiment Name:", style=style, layout=layout_half)


# Display widgets
display(Markdown("## Stage 1: Data Ingestion Parameters"))
display(api_key_input, bible_id_input, text_fileset_id_input, books_to_download_input, data_dir_widget)

display(Markdown("## Stage 2: ASR Fine-tuning Parameters"))
display(ssl_ckpt_widget, epochs_widget, asr_experiment_name_widget)

# Store widget values for later use (initial values)
WIDGET_VALUES = {
    'api_key': api_key_input.value,
    'bible_id': bible_id_input.value,
    'text_fileset_id': text_fileset_id_input.value,
    'books': books_to_download_input.value,
    'data_dir': data_dir_widget.value,
    'ssl_ckpt': ssl_ckpt_widget.value,
    'epochs': epochs_widget.value,
    'asr_experiment_name': asr_experiment_name_widget.value
}

def update_widget_values(b=None): # b is dummy for button events
    WIDGET_VALUES['api_key'] = api_key_input.value
    WIDGET_VALUES['bible_id'] = bible_id_input.value
    WIDGET_VALUES['text_fileset_id'] = text_fileset_id_input.value
    WIDGET_VALUES['books'] = books_to_download_input.value
    WIDGET_VALUES['data_dir'] = data_dir_widget.value
    WIDGET_VALUES['ssl_ckpt'] = ssl_ckpt_widget.value
    WIDGET_VALUES['epochs'] = epochs_widget.value
    WIDGET_VALUES['asr_experiment_name'] = asr_experiment_name_widget.value
    # Create the data directory if it doesn't exist
    Path(WIDGET_VALUES['data_dir']).mkdir(parents=True, exist_ok=True)
    print("Widget values captured/updated.")

# Update values when a field is changed or a button is clicked later
api_key_input.observe(update_widget_values, names='value')
bible_id_input.observe(update_widget_values, names='value')
text_fileset_id_input.observe(update_widget_values, names='value')
books_to_download_input.observe(update_widget_values, names='value')
data_dir_widget.observe(update_widget_values, names='value')
ssl_ckpt_widget.observe(update_widget_values, names='value')
epochs_widget.observe(update_widget_values, names='value')
asr_experiment_name_widget.observe(update_widget_values, names='value')

update_widget_values() # Initial capture

In [ ]:
def run_command(command_list, cwd=None, live_output=True):
    """Runs a shell command (list of args) and prints its output."""
    if cwd is None:
        cwd = PROJECT_ROOT # Ensure commands run from project root by default
    
    command_str = " ".join(command_list)
    display(Markdown(f"**Running Command:**\n```bash\n{command_str}\n```\nIn directory: `{cwd}`"))
    
    process = subprocess.Popen(command_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, universal_newlines=True, cwd=cwd)
    
    output_lines = []
    if live_output:
        for line in iter(process.stdout.readline, ''):
            print(line, end='') # Print in real-time
            output_lines.append(line)
        process.stdout.close()
    
    return_code = process.wait()
    
    if not live_output: # If not live, print all output at the end
        stdout, _ = process.communicate()
        print(stdout)
        output_lines = stdout.splitlines()

    if return_code == 0:
        display(Markdown(f"<font color='green'>Command completed successfully (Code: {return_code}).</font>"))
    else:
        display(Markdown(f"<font color='red'>Command failed with error code {return_code}.</font>"))
        # if not live_output: print("".join(output_lines)) # Print output if it wasn't live

    return return_code, "".join(output_lines)

In [ ]:
# --- Download Bible Data ---
display(Markdown("### 1. Download Bible Audio and Text"))
download_button = widgets.Button(description="Start Download")
download_output = widgets.Output()

def download_data_action(b):
    with download_output:
        download_output.clear_output(wait=True)
        update_widget_values() # Ensure latest values are used
        
        if not WIDGET_VALUES['api_key']:
            display(Markdown("<font color='red'>DBP API Key is required!</font>"))
            return

        cmd = [
            sys.executable, str(DOWNLOAD_SCRIPT_PATH),
            "--api_key", WIDGET_VALUES['api_key'],
            "--dest", WIDGET_VALUES['data_dir'],
            "--language_codes", "nyo", # Focus on Runyoro for this fine-tuning
            "--fileset_ids_audio", WIDGET_VALUES['bible_id'], # Audio fileset ID for Runyoro
            "--fileset_id_text", WIDGET_VALUES['text_fileset_id'],   # Text fileset ID for Runyoro
        ]
        if WIDGET_VALUES['books']:
            cmd.extend(["--book_ids", WIDGET_VALUES['books']])
        
        run_command(cmd)

download_button.on_click(download_data_action)
display(download_button, download_output)

In [ ]:
# --- Build Manifest ---
display(Markdown("### 2. Build SpeechBrain JSON Manifest"))
manifest_button = widgets.Button(description="Create Manifest")
manifest_output = widgets.Output()

# Define manifest filename based on Bible ID and books
# Ensure this path is within the DATA_DIR
MANIFEST_FILENAME = f"manifest_sb_{WIDGET_VALUES.get('bible_id', 'custom')}_{WIDGET_VALUES.get('books', 'all').replace(',', '_')}.json"
MANIFEST_FILE_PATH = Path(WIDGET_VALUES.get('data_dir', DEFAULT_DATA_DIR)) / MANIFEST_FILENAME


def update_manifest_path(b=None):
    global MANIFEST_FILE_PATH, MANIFEST_FILENAME
    update_widget_values()
    MANIFEST_FILENAME = f"manifest_sb_{WIDGET_VALUES['bible_id']}_{WIDGET_VALUES['books'].replace(',', '_') if WIDGET_VALUES['books'] else 'all'}.json"
    MANIFEST_FILE_PATH = Path(WIDGET_VALUES['data_dir']) / MANIFEST_FILENAME
    print(f"Manifest file will be: {MANIFEST_FILE_PATH}")

# Update manifest path when relevant widgets change
bible_id_input.observe(update_manifest_path, names='value')
books_to_download_input.observe(update_manifest_path, names='value')
data_dir_widget.observe(update_manifest_path, names='value')
update_manifest_path() # Initial call


def build_manifest_action(b):
    with manifest_output:
        manifest_output.clear_output(wait=True)
        update_widget_values() # Ensure latest values
        update_manifest_path() # Update manifest path based on current values

        audio_data_path = Path(WIDGET_VALUES['data_dir']) / "audio"
        text_data_path = Path(WIDGET_VALUES['data_dir']) / "text"

        cmd = [
            sys.executable, str(MANIFEST_SCRIPT_PATH),
            "--audio_dir_base", str(audio_data_path),
            "--text_dir_base", str(text_data_path),
            "--audio_fileset_id", WIDGET_VALUES['bible_id'],
            "--text_fileset_id", WIDGET_VALUES['text_fileset_id'],
            "--manifest_path", str(MANIFEST_FILE_PATH),
            "--language_code", "nyo" 
        ]
        run_command(cmd)
        if MANIFEST_FILE_PATH.exists():
            display(Markdown(f"Manifest created: `{MANIFEST_FILE_PATH}`"))
            # Display first few lines of manifest
            with open(MANIFEST_FILE_PATH, 'r') as f:
                content_preview = "".join([next(f) for _ in range(10)])
            display(Markdown(f"**Manifest Preview (first 10 lines):**
```json
{content_preview}
...```"))
        else:
            display(Markdown(f"<font color='red'>Manifest creation failed. File not found: {MANIFEST_FILE_PATH}</font>"))


manifest_button.on_click(build_manifest_action)
display(manifest_button, manifest_output)

In [ ]:
# --- Prepare Hyperparameters for Training ---
# The train_ctc.py script expects a YAML file for hyperparameters.
# We will generate this YAML dynamically based on widget inputs and defaults.
import yaml
import torch # For device check

display(Markdown("### 3. Prepare Hyperparameters for ASR Training"))

# Define where the hparams file for this experiment will be saved
ASR_EXPERIMENT_DIR = DEFAULT_ASR_OUTPUT_DIR / WIDGET_VALUES.get('asr_experiment_name', 'default_asr_run')
HPARAMS_FILE = ASR_EXPERIMENT_DIR / "hparams_finetune.yaml"

def generate_hparams_yaml():
    update_widget_values() # Ensure WIDGET_VALUES are current
    update_manifest_path() # Ensure MANIFEST_FILE_PATH is current

    # Use the current value from the widget for ASR_EXPERIMENT_DIR path construction
    current_asr_experiment_name = WIDGET_VALUES['asr_experiment_name']
    current_asr_experiment_dir = DEFAULT_ASR_OUTPUT_DIR / current_asr_experiment_name
    current_asr_experiment_dir.mkdir(parents=True, exist_ok=True)
    current_hparams_file = current_asr_experiment_dir / "hparams_finetune.yaml"

    hparams_content = {
        "seed": 1234,
        "data_folder": WIDGET_VALUES['data_dir'], 
        "output_folder": str(current_asr_experiment_dir), 
        "save_folder": "!ref <output_folder>/save", 
        "device": "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else "cpu"),

        "train_manifest": str(MANIFEST_FILE_PATH.relative_to(Path(WIDGET_VALUES['data_dir']))), 
        "valid_manifest": str(MANIFEST_FILE_PATH.relative_to(Path(WIDGET_VALUES['data_dir']))), 
        "test_manifest": str(MANIFEST_FILE_PATH.relative_to(Path(WIDGET_VALUES['data_dir']))),  

        "tokenizer_model_dir": "!ref <save_folder>/tokenizer/", 
        "tokenizer_model_prefix": "spm_unigram_1000", 
        "tokenizer_vocab_size": 1000, 
        "tokenizer_model_type": "unigram", 
        "tokenizer_char_coverage": 0.9995, 
        "force_retrain_tokenizer": False, 

        "sample_rate": 16000, 
        "max_batch_len_train_s": 20.0, 
        "max_batch_len_valid_s": 15.0,
        "max_batch_len_test_s": 15.0,
        "num_buckets_train": 30,
        "num_buckets_valid": 10,
        "num_buckets_test": 10,
        "train_dataloader_opts": {"num_workers": 2 if IN_COLAB else 4, "pin_memory": True},
        "valid_dataloader_opts": {"num_workers": 2, "pin_memory": True},
        "test_dataloader_opts": {"num_workers": 2, "pin_memory": True},

        "ssl_model_hub": "facebook/wav2vec2-base", 
        "ssl_model_output_dim": 768, 
        "ssl_model_requires_grad": False, 
        "freeze_ssl_epochs": 1, 
        
        "asr_model_class_name": "CRDNN", 
        "asr_model_hparams": { 
            "huggingface_cache_dir": str(PROJECT_ROOT / ".cache" / "huggingface_models"), 
            "ssl_output_norm": True, 
            "rnn_class": "!name:torch.nn.LSTM",
            "rnn_layers": 4, "rnn_neurons": 512, "rnn_bidirectional": True, "rnn_dropout": 0.15,
            "dnn_blocks": 2, "dnn_neurons": 512, "dnn_activation": "!name:torch.nn.ReLU", "dnn_dropout": 0.15,
        },
        "asr_output_features": 512, 

        "ctc_output_neurons": "!ref <tokenizer_vocab_size> + 1", 

        "lr": 3e-4,
        "lr_scheduler": "!new:speechbrain.nnet.schedulers.NewBobScheduler", 
        "lr_scheduler_params": {"initial_value": "!ref <lr>", "improvement_threshold": 0.0025, "annealing_factor": 0.9, "patience": 2},
        "lr_scheduler_monitor_key": "WER", 
        "amsgrad": False,

        "epoch_counter": "!new:speechbrain.utils.epoch_loop.EpochCounter",
        "epoch_counter_params": {"limit": WIDGET_VALUES['epochs']},
        "ddp_no_sync": False, 

        "checkpoint_best_key": "WER", 
        "save_folder_format": "epoch_{epoch:03d}", 

        "log_softmax": "!name:torch.nn.LogSoftmax", "log_softmax_params": {"dim": -1},
        "ctc_loss": "!name:speechbrain.nnet.losses.ctc_loss",
        "ctc_loss_params": {"blank_index": 0}, 
        "blank_index": 0, 
        "bos_index": 1, 
        "eos_index": 2, 
        "cer_computer": "!name:speechbrain.utils.metric_stats.ErrorRateStats", "cer_computer_params": {"split_tokens": True},
        "error_rate_computer": "!name:speechbrain.utils.metric_stats.ErrorRateStats",
        "wer_file": "!ref <output_folder>/wer_test_results.txt", 

        "ssl_local_checkpoint_path": WIDGET_VALUES['ssl_ckpt'],
    }

    with open(current_hparams_file, 'w') as f:
        yaml.dump(hparams_content, f, sort_keys=False, indent=2)
    
    display(Markdown(f"Hyperparameters file generated: `{current_hparams_file}`"))
    display(Markdown(f"**Content:**
```yaml
    f"{yaml.dump(hparams_content, sort_keys=False, indent=2)}
```"))
    return current_hparams_file


# Generate HParams button
hparams_button = widgets.Button(description="Generate HParams YAML")
hparams_output = widgets.Output()

def hparams_action(b):
    with hparams_output:
        hparams_output.clear_output(wait=True)
        generate_hparams_yaml()

hparams_button.on_click(hparams_action)
display(hparams_button, hparams_output)
# Auto-generate on load for convenience if values are already set
if Path(WIDGET_VALUES['data_dir']).exists() and MANIFEST_FILE_PATH.exists(): # Basic check
    with hparams_output:
      hparams_output.clear_output(wait=True)
      print("Auto-generating hparams based on current widget values...")
      generate_hparams_yaml()
else:
    print("Run data download and manifest creation first, then click 'Generate HParams YAML'.")


In [ ]:
# --- Run ASR Fine-tuning ---
display(Markdown("### 4. Run ASR Fine-tuning Training"))
train_button = widgets.Button(description="Start ASR Training")
train_output = widgets.Output()

def train_asr_action(b):
    with train_output:
        train_output.clear_output(wait=True)
        update_widget_values() # Ensure latest values
        
        # Regenerate hparams to capture any last-minute widget changes
        current_hparams_file = generate_hparams_yaml()
        if not current_hparams_file or not current_hparams_file.exists():
            display(Markdown("<font color='red'>Hyperparameter YAML file not found. Please generate it first.</font>"))
            return

        ssl_checkpoint_path = WIDGET_VALUES['ssl_ckpt']
        if ssl_checkpoint_path and not Path(ssl_checkpoint_path).exists():
             display(Markdown(f"<font color='red'>Warning: Specified SSL Checkpoint path does not exist: {ssl_checkpoint_path}. Training might fail if the script expects it.</font>"))


        cmd = [
            sys.executable, str(TRAIN_SCRIPT_PATH),
            str(current_hparams_file),
        ]
        
        display(Markdown("Training may take a very long time and requires a GPU for reasonable speed."))
        run_command(cmd)
        
        current_asr_experiment_name = WIDGET_VALUES['asr_experiment_name']
        final_model_dir = DEFAULT_ASR_OUTPUT_DIR / current_asr_experiment_name / "save"
        display(Markdown(f"Training finished. Checkpoints should be in `{final_model_dir}`."))
        display(Markdown(f"Look for files like `CKPT+epoch-{WIDGET_VALUES['epochs']}.pth` or similar, depending on SpeechBrain's checkpointer naming."))


train_button.on_click(train_asr_action)
display(train_button, train_output)

In [ ]:
# --- Decode 3 Random Test Clips ---
import torch
import torchaudio
from IPython.display import Audio # For playing audio in notebook
import yaml # For loading hparams
from speechbrain.tokenizers.SentencePiece import SentencePiece # Ensure this is the correct import path

display(Markdown("### 5. Decode Sample Audio with Fine-tuned Model"))
decode_button = widgets.Button(description="Decode Samples")
decode_output = widgets.Output()

def decode_samples_action(b):
    with decode_output:
        decode_output.clear_output(wait=True)
        update_widget_values()
        update_manifest_path() # for MANIFEST_FILE_PATH
        
        display(Markdown("Decoding samples requires a trained model checkpoint. "
                         "This part is a simplified demonstration and might need adjustments based on "
                         "how `train_ctc.py` saves its final model and tokenizer."))

        try:
            current_asr_experiment_name = WIDGET_VALUES['asr_experiment_name']
            hparams_for_inference_path = DEFAULT_ASR_OUTPUT_DIR / current_asr_experiment_name / "hparams_finetune.yaml"
            
            if not hparams_for_inference_path.exists():
                display(Markdown(f"<font color='red'>HParams file for inference not found at {hparams_for_inference_path}. Cannot proceed.</font>"))
                return

            with open(hparams_for_inference_path) as fin:
                hparams_inf = yaml.load(fin, Loader=yaml.SafeLoader) 

            tokenizer_model_dir_inf = Path(hparams_inf["output_folder"]) / Path(hparams_inf["tokenizer_model_dir"]).name 
            if not tokenizer_model_dir_inf.is_absolute() and not tokenizer_model_dir_inf.exists():
                 tokenizer_model_dir_inf = Path(hparams_inf["output_folder"]) / hparams_inf["tokenizer_model_dir"]

            tokenizer_path_inf = tokenizer_model_dir_inf / (hparams_inf["tokenizer_model_prefix"] + ".model")
            
            if not tokenizer_path_inf.exists():
                 display(Markdown(f"<font color='red'>Tokenizer model not found at {tokenizer_path_inf}. Cannot proceed with decoding.</font>"))
                 return

            tokenizer_inf = SentencePiece(str(tokenizer_path_inf), model_type=hparams_inf["tokenizer_model_type"])
            display(Markdown(f"Tokenizer loaded successfully from {tokenizer_path_inf}"))

            if not MANIFEST_FILE_PATH.exists():
                display(Markdown(f"<font color='red'>Manifest file {MANIFEST_FILE_PATH} not found. Cannot select samples.</font>"))
                return
            
            with open(MANIFEST_FILE_PATH, 'r') as f:
                manifest_data = json.load(f)
            
            if not manifest_data:
                display(Markdown("<font color='red'>Manifest data is empty.</font>"))
                return

            all_utt_ids = list(manifest_data.keys())
            import random
            sample_ids = random.sample(all_utt_ids, min(3, len(all_utt_ids)))

            if not sample_ids:
                display(Markdown("<font color='red'>No samples to decode.</font>"))
                return

            display(Markdown("--- Sampled Audio Files for Decoding ---"))
            for utt_id in sample_ids:
                audio_path = manifest_data[utt_id]["wav"]
                true_text = manifest_data[utt_id]["words"]
                duration = manifest_data[utt_id]["duration"]
                
                display(Markdown(f"**Utterance ID:** {utt_id}"))
                display(Markdown(f"**Audio Path:** `{audio_path}` (Duration: {duration:.2f}s)"))
                display(Markdown(f"**True Transcript:** {true_text}"))
                display(Audio(audio_path))
                display(Markdown(f"**Predicted Transcript:** (Decoding logic not fully implemented in this notebook template. "
                                 "A separate inference script using the trained model is recommended.)"))
                display(Markdown("---"))
            
            display(Markdown("To perform actual decoding, you would typically:\n"
                             "1. Initialize your `CustomASRModel` and `ctc_output_layer` as defined in `train_ctc.py`.\n"
                             "2. Load the saved checkpoint weights into these modules (e.g., using `sb.utils.checkpoints.Checkpointer.load_checkpoint`).\n"
                             "3. For each audio file: load it, pass it through the model, apply softmax, and use a CTC decoder (greedy or beam search).\n"
                             "This often involves reusing parts of the `CTCBrain`'s `evaluate_batch` logic or SpeechBrain's `EncoderASR.transcribe_file` if applicable."))

        except Exception as e:
            display(Markdown(f"<font color='red'>Error during sample decoding setup: {e}</font>"))
            import traceback
            traceback.print_exc()

decode_button.on_click(decode_samples_action)
display(decode_button, decode_output)